# Verify TimeSeriesDataset

In [1]:
import sys
sys.path.insert(0, '/Users/shelleygoel/Code/01_statistical_mod_blog/TSB-AD/explorations')
sys.path.insert(0, '/Users/shelleygoel/Code/01_statistical_mod_blog/anomaly_detection')

import hvac_data_gen as hvdg
from core.dataset import TimeSeriesDataset
from datetime import datetime
import pandas as pd
import numpy as np

In [2]:
# Generate test data with lag anomaly
gen = hvdg.HVACDataGenerator(seed=42)

lag_config = [{
    'unit': 1, 'type': 'lag',
    'start_day': 2, 'start_hour': 8,
    'duration_hours': 48,
    'params': {'lag_minutes': 180}
}]
df_lag = gen.generate_container_data(
    container_id=0, start_time=datetime(2026, 1, 15),
    duration_days=5, anomaly_config=lag_config
)

# Normal container
df_normal = gen.generate_container_data(
    container_id=1, start_time=datetime(2026, 1, 15),
    duration_days=5, anomaly_config=[]
)

hvac_df = pd.concat([df_lag, df_normal], ignore_index=True)
print(f"Shape: {hvac_df.shape}")
print(f"Columns: {list(hvac_df.columns)}")
hvac_df.head()

Shape: (43200, 6)
Columns: ['timestamp_et', 'HVACNum', 'TmpRet', 'anomaly', 'anomaly_type', 'container_id']


,timestamp_et,HVACNum,TmpRet,anomaly,anomaly_type,container_id
0,2026-01-15 00:00:00,0,49.585463,False,normal,0
1,2026-01-15 00:00:00,1,50.828930,False,normal,0
2,2026-01-15 00:00:00,2,50.299590,False,normal,0
3,2026-01-15 00:01:00,0,49.531567,False,normal,0
4,2026-01-15 00:01:00,1,50.915807,False,normal,0


In [3]:
# Create dataset
# Adapt col names to what hvac_data_gen actually produces
print("Unique anomaly_type values:", hvac_df['anomaly_type'].unique())
print("Unique unit values:", hvac_df['unit'].unique() if 'unit' in hvac_df.columns else 'NO unit col')
print("Unique HVACNum values:", hvac_df['HVACNum'].unique() if 'HVACNum' in hvac_df.columns else 'NO HVACNum col')

Unique anomaly_type values: <ArrowStringArray>
['normal', 'lag']
Length: 2, dtype: str
Unique unit values: NO unit col
Unique HVACNum values: [0 1 2]


In [4]:
hvac_df['humidity'] = np.random.uniform(30, 40, len(hvac_df))

In [5]:
hvac_df.head()

,timestamp_et,HVACNum,TmpRet,anomaly,anomaly_type,container_id,humidity
0,2026-01-15 00:00:00,0,49.585463,False,normal,0,33.943878
1,2026-01-15 00:00:00,1,50.828930,False,normal,0,33.674344
2,2026-01-15 00:00:00,2,50.299590,False,normal,0,37.032612
3,2026-01-15 00:01:00,0,49.531567,False,normal,0,39.902232
4,2026-01-15 00:01:00,1,50.915807,False,normal,0,34.053538


# TODO:
- Test plotting of multivariate Timeseries, color coded by subentity
- Create an EDA class - which has some standard figures for the dataset.

In [6]:
# Determine correct column names from data
sub_entity_col = 'unit' if 'unit' in hvac_df.columns else 'HVACNum'
label_col = 'anomaly' if 'anomaly' in hvac_df.columns else 'label'

col_map = {
    'entity': 'container_id',
    'time': 'timestamp_et',
    'value_cols': ['TmpRet', 'humidity'],
    'label': label_col,
    'label_type': 'anomaly_type',
    'sub_entity': sub_entity_col,
}

ds = TimeSeriesDataset(hvac_df, col_map)
print("Created successfully!")

Created successfully!


In [7]:
# Test entities()

# Test anomaly_types()
print("Anomaly types:", ds.anomaly_types())
assert 'normal' in ds.anomaly_types()
assert 'lag' in ds.anomaly_types()

Anomaly types: <ArrowStringArray>
['normal', 'lag']
Length: 2, dtype: str


In [8]:
ds.anomaly_summary()

,label_type,entity_count
0,lag,1
1,normal,1


In [9]:
# Test day_labels()
dl = ds.day_labels()
print("day_labels columns:", list(dl.columns))
print(dl)

# Container 0 should have anomaly on days 2-3 (lag starts day 2 hour 8, 48h)
# Container 1 should have no anomalies
c0_labels = dl[dl['container_id'] == 0]
c1_labels = dl[dl['container_id'] == 1]
print("\nContainer 0 anomaly days:", c0_labels[c0_labels['anomaly'] > 0]['day'].tolist())
print("Container 1 anomaly days:", c1_labels[c1_labels['anomaly'] > 0]['day'].tolist())

day_labels columns: ['container_id', 'day', 'anomaly', 'anomaly_type']
   container_id         day  anomaly anomaly_type
0             0  2026-01-15    False       normal
1             0  2026-01-16    False       normal
2             0  2026-01-17     True          lag
3             0  2026-01-18     True          lag
4             0  2026-01-19     True          lag
5             1  2026-01-15    False       normal
6             1  2026-01-16    False       normal
7             1  2026-01-17    False       normal
8             1  2026-01-18    False       normal
9             1  2026-01-19    False       normal

Container 0 anomaly days: [datetime.date(2026, 1, 17), datetime.date(2026, 1, 18), datetime.date(2026, 1, 19)]
Container 1 anomaly days: []


In [10]:
# Test ts_labels()
tl = ds.ts_labels()
print("ts_labels columns:", list(tl.columns))
print("ts_labels shape:", tl.shape)
print("\nAnomaly timestamps (first 5):")
print(tl[tl['anomaly'] > 0].head())

ts_labels columns: ['container_id', 'timestamp_et', 'anomaly', 'anomaly_type']
ts_labels shape: (14400, 4)

Anomaly timestamps (first 5):
      container_id        timestamp_et  anomaly anomaly_type
3360             0 2026-01-17 08:00:00     True          lag
3361             0 2026-01-17 08:01:00     True          lag
3362             0 2026-01-17 08:02:00     True          lag
3363             0 2026-01-17 08:03:00     True          lag
3364             0 2026-01-17 08:04:00     True          lag


In [11]:
# Test sample_and_visualize_cases
figs = ds.sample_and_visualize_cases(n_cases=2, label_type='lag')
for fig in figs:
    fig.show()

In [17]:
# Validation: check col_map validation catches errors
try:
    TimeSeriesDataset(hvac_df, {'entity': 'container_id'})  # missing required keys
except ValueError as e:
    print(f"Caught expected error: {e}")

try:
    bad_map = {**col_map, 'bogus': 'foo'}
    TimeSeriesDataset(hvac_df, bad_map)  # unknown key
except ValueError as e:
    print(f"Caught expected error: {e}")

try:
    bad_map = {**col_map, 'entity': 'nonexistent_col'}
    TimeSeriesDataset(hvac_df, bad_map)  # missing column
except ValueError as e:
    print(f"Caught expected error: {e}")

Caught expected error: col_map missing required keys: {'value_cols', 'time'}
Caught expected error: col_map has unknown keys: {'bogus'}
Caught expected error: Columns not found in DataFrame: ['nonexistent_col']


# Model Class Test
- test model class works with dataset class
- and outputs a score
- 